# Summer 2025 URI Spotted Lanternfly Final Predictive Model Report
## Erica Keklak | 2025-06-25 to present
----
This is the final, simplified model report showing model development and data visualizations beginning after the training dataset was finished.

For future reference, the processing of data happened in the following steps:
1. Geometry data was processed in the notebook "Processing Geometry Data for Machine Learning Models"
2. Spotted lanternfly abundance/infestation data was processed in the notebook "Processing SLF Observation Data for Machine Learning Models"
3. Host plant abundance/presence data was processed in the notebook "Processing Host Plant Data for Machine Learning Models"
4. Traffic (both primary road and railway) data was processed in the notebook "Processing Traffic Data for Machine Learning Models"
5. Climate data was processed in the notebook "Processing Local Climatological Data for Machine Learning Models"
6. Predator abundance/presence data was processed in the notebook "Processing Predator Data for Machine Learning Models"
7. Target classes were assigned in the notebook "Adding Target Classes for Machine Learning Models"

In [119]:
# Packages and libraries for data processing (cleaning, preparation, and getting basic statistics from them)

import numpy as np
import pandas as pd
import math as m
import random as rand
import pickle

# Packages and libraries for primarily statistical data visualization

import matplotlib.pyplot as plt
import seaborn as sn

# Packages and libraries for primarily geospatial data visualization

import geopandas as gpd # originally had to try installing to my device using the Anaconda Powershell Prompt command
                        # conda install -c conda-forge geopandas
from shapely.geometry import Point

# Packages, functions, and libraries for model development

from sklearn.model_selection import cross_val_score, StratifiedKFold, train_test_split # for test splitting and model verification
from sklearn.ensemble import RandomForestClassifier # for a random forest classification method
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler, OrdinalEncoder # encoding is needed to properly process data by algorithms
from sklearn.tree import DecisionTreeClassifier # another decision tree method
from sklearn.metrics import classification_report, confusion_matrix # getting a report from a classification model

## Model Development: General Spotted Lanternfly Spread Risk Model

In [65]:
# Import the datasets that were used

county_geom = gpd.read_file('C:/Users/EK111/Documents/NJIT/Spotted Lanternfly Project URI Summer 2025/Modified Data Backups/Geometry/county_geometry.shp')
state_geom = gpd.read_file('C:/Users/EK111/Documents/NJIT/Spotted Lanternfly Project URI Summer 2025/Modified Data Backups/Geometry/state_geometry.shp')
county_gdf = gpd.read_file('C:/Users/EK111/Documents/NJIT/Spotted Lanternfly Project URI Summer 2025/Modified Data Backups/Training/general_county_training.shp')
state_gdf = gpd.read_file('C:/Users/EK111/Documents/NJIT/Spotted Lanternfly Project URI Summer 2025/Modified Data Backups/Training/general_state_training.shp')
# food_county_gdf = gpd.read_file()
# food_state_gdf = gpd.read_file()
# fiber_county_gdf = gpd.read_file()
# fiber_state_gdf = gpd.read_file()
# ornamental_county_gdf = gpd.read_file()
# ornamental_state_gdf = gpd.read_file()

In [111]:
def split_50_30_20_new(df: pd.DataFrame, is_for: int = 2025, seed = 1) -> dict:
    years = {}
    unique_years = list(pd.unique(df['year']))
    for i in range(0, len(unique_years)):
        years[int(unique_years[i])] = df.loc[df['year'] == unique_years[i]]

    leave_in = [x for x in unique_years if x not in list(range(2025, is_for + 1))]
    leave_out = [x for x in unique_years if x in list(range(2025, is_for + 1))]
    leave_in_df = df.loc[df['year'] == unique_years[0]]
    for i in range(1, len(unique_years)):
        this_year = unique_years[i]
        this_year_df = df.loc[df['year'] == this_year]
        leave_in_df = pd.concat([leave_in_df, this_year_df], axis = 0)
        
    leave_in_x = leave_in_df.drop(columns = ['geometry', 'target'], axis = 1)
    leave_in_y = leave_in_df['target']

    basic_x = pd.DataFrame({'county': [], 'state': [], 'year': [], 'slf_pop': [], 'slfdensity': []})
    X_train = basic_x
    X_validation = basic_x
    X_test = basic_x

    basic_y = pd.DataFrame({'target': []})
    Y_train = basic_y
    Y_validation = basic_y
    Y_test = basic_y
    
    for i in range(0, len(leave_in)):
        X_train_now, X_other_now, Y_train_now, Y_other_now = train_test_split(leave_in_x, leave_in_y, train_size = 0.5, shuffle = True,
                                                                              random_state = seed
                                                                             ) # 50% of all data does into training
        X_validation_now, X_test_now, Y_validation_now, Y_test_now = train_test_split(X_other_now, Y_other_now, train_size = 0.6, shuffle = True,
                                                                                      random_state = seed
                                                                                     ) # 0.6 is 60% of the remaining data, or 30% of total data, which
                                                                                       # represents the preferred amount for validation
        X_train = pd.concat([X_train, X_train_now], axis = 0, join = 'outer')
        X_validation = pd.concat([X_validation, X_validation_now], axis = 0, join = 'outer')
        X_test = pd.concat([X_test, X_test_now], axis = 0, join = 'outer')
        Y_train = pd.concat([Y_train, Y_train_now], axis = 0, join = 'outer')
        Y_validation = pd.concat([Y_validation, Y_validation_now], axis = 0, join = 'outer')
        Y_test = pd.concat([Y_test, Y_test_now], axis = 0, join = 'outer')

    train_set = X_train
    train_set['target'] = Y_train['target']

    validation_set = X_validation
    validation_set['target'] = Y_validation['target']

    test_set = X_test
    test_set['target'] = Y_test['target']

    predict_set = df.loc[df['year'] == is_for]
    if len(leave_out) > 1:
        for i in range(0, len(leave_out)):
            this_year = leave_out[i]
            this_year_df = df.loc[df['year'] == this_year]
            predict_set = pd.concat([predict_set, this_year_df], axis = 0)
    
    # predict_set = pd.DataFrame(df.loc[df['year'] not in leave_in])
    X_predict = predict_set.drop(columns = ['geometry', 'target'], axis = 1)

    return {'X_train': X_train, 'X_validation': X_validation, 'X_test': X_test, 'X_predict': X_predict, 'Y_train': Y_train, 'Y_validation': Y_validation,
            'Y_test': Y_test, 'training data': train_set, 'validation data': validation_set, 'test data': test_set, 'data to predict': predict_set}
    #     this_year_index = list(this_year.index)
    #     shuffled_indices = rand.shuffle(this_year_index)

In [141]:
county_gdf_2025 = county_geom
county_gdf_2025['year'] = [2025] * len(county_geom)

county_gdf_2026 = county_geom
county_gdf_2026['year'] = [2026] * len(county_geom)

county_training_2025 = pd.concat([county_gdf, county_gdf_2025], join = 'outer', axis = 0)
# county_training_2025['index'] = np.arange(len(county_training_2025))
# county_training_2025 = county_training_2025.set_index('index')
county_training_2025 = county_training_2025.reset_index(drop = True)

county_training_2026 = pd.concat([county_training_2025, county_gdf_2026], join = 'outer', axis = 0)
# county_training_2026['index'] = np.arange(len(county_training_2025))
# county_training_2026 = county_training_2026.set_index('index')
county_training_2026 = county_training_2026.reset_index(drop = True)

state_gdf_2025 = state_geom
state_gdf_2025['year'] = [2025] * len(state_geom)

state_gdf_2026 = state_geom
state_gdf_2026['year'] = [2026] * len(state_geom)

state_training_2025 = pd.concat([state_gdf, state_gdf_2025], join = 'outer', axis = 0)
# state_training_2025['index'] = np.arange(len(state_training_2025))
# state_training_2025 = state_training_2025.set_index('index')
state_training_2025 = state_training_2025.reset_index(drop = True)

state_training_2026 = pd.concat([state_training_2025, state_gdf_2026], join = 'outer', axis = 0)
# state_training_2026['index'] = np.arange(len(state_training_2026))
# state_training_2026 = state_training_2026.set_index('index')
state_training_2026 = state_training_2026.reset_index(drop = True)

In [143]:
c5_copy = county_training_2025.copy()
c6_copy = county_training_2026.copy()
s5_copy = state_training_2025.copy()
s6_copy = state_training_2026.copy()

In [145]:
c5_copy.at[1, 'county']

'Choctaw'

In [147]:
c5_encoded = my_encoder_v2(c5_copy)
c5_encoded = my_encoder_v2(c5_copy)
c5_encoded = my_encoder_v2(c5_copy)
c5_encoded = my_encoder_v2(c5_copy)

Processing the column county


ValueError: Expected 2D array, got 1D array instead:
array=['Houston' 'Choctaw' 'Russell' ... 'Webster' 'Phelps' 'Audrain'].
Reshape your data either using array.reshape(-1, 1) if your data has a single feature or array.reshape(1, -1) if it contains a single sample.

In [115]:
c5_split = split_50_30_20_new(c5_copy, is_for = 2025, seed = 1)
c6_split = split_50_30_20_new(c6_copy, is_for = 2025, seed = 1)
s5_split = split_50_30_20_new(s5_copy, is_for = 2025, seed = 1)
s6_split = split_50_30_20_new(s6_copy, is_for = 2025, seed = 1)

In [117]:
general_county_2025_model = DecisionTreeClassifier()

general_county_2025_model.fit(c5_split['X_train'], c5_split['Y_train'])
    
cv_results = cross_val_score(RandomForestClassifier(n_estimators = 100), c5_split['X_train'], c5_split['Y_train'], cv = 5, scoring = 'accuracy')
print('Average training-validation accuracy: ',cv_results.mean(),' Standard deviation of training-validation accuracy: ',cv_results.std())
    
print(classification_report(c5_split['Y_validation'], general_county_2025_model.predict(c5_split['X_validation'])))
print(confusion_matrix(c5_split['Y_validation'], general_county_2025_model.predict(c5_split['X_validation'])))
    
tt_results = cross_val_score(RandomForestClassifier(n_estimators = 100), c5_split['X_test'], c5_split['Y_test'], cv = 5, scoring = 'accuracy')
print('Average training-test accuracy: ',tt_results.mean(),' Standard deviation of training-test accuracy: ',tt_results.std())

print(classification_report(c5_split['Y_test'], general_county_2025_model.predict(c5_split['X_test'])))
print(confusion_matrix(c5_split['Y_test'], general_county_2025_model.predict(c5_split['X_test'])))

print('Predictions for counties in 2025:')

general_county_2025_predictions = general_county_2025_model.predict(c5_split['X_predict'])

general_county_2025_predictions_df = c5_split['X_predict']
general_county_2025_predictions_df['target'] = general_county_2025_predictions

general_county_2025_predictions_df.head()

ValueError: could not convert string to float: 'Nicollet'

In [ ]:
# Save these models using Pickle

# models = 'C:/Users/EK111/Documents/NJIT/Spotted Lanternfly Project URI Summer 2025/Deliverables/During Research/Models/'
# path_grfc5 = str(models + 'general_spread_risk_county_rf_model_2025_NOENCODE.sav')
pickle.dump(general_county_2025_model, open('C:/Users/EK111/Documents/NJIT/Spotted Lanternfly Project URI Summer 2025/Deliverables/During Research/Models/county_updated_model_2025.sav', 'wb'))

In [ ]:
general_state_2025_model = DecisionTreeClassifier()

general_state_2025_model.fit(s5_split['X_train'], s5_split['Y_train'])
    
cv_results = cross_val_score(RandomForestClassifier(n_estimators = 100), s5_split['X_train'], s5_split['Y_train'], cv = 5, scoring = 'accuracy')
print('Average training-validation accuracy: ',cv_results.mean(),' Standard deviation of training-validation accuracy: ',cv_results.std())
    
print(classification_report(s5_split['Y_validation'], general_state_2025_model.predict(s5_split['X_validation'])))
print(confusion_matrix(s5_split['Y_validation'], general_state_2025_model.predict(s5_split['X_validation'])))
    
tt_results = cross_val_score(RandomForestClassifier(n_estimators = 100), s5_split['X_test'], s5_split['Y_test'], cv = 5, scoring = 'accuracy')
print('Average training-test accuracy: ',tt_results.mean(),' Standard deviation of training-test accuracy: ',tt_results.std())

print(classification_report(s5_split['Y_test'], general_state_2025_model.predict(s5_split['X_test'])))
print(confusion_matrix(s5_split['Y_test'], general_state_2025_model.predict(s5_split['X_test'])))

print('Predictions for states in 2025:')

general_state_2025_predictions = general_state_2025_model.predict(s5_split['X_predict'])

general_state_2025_predictions_df = s5_split['X_predict']
general_state_2025_predictions_df['target'] = general_state_2025_predictions

general_county_2025_predictions_df.head()

In [ ]:
# path_grfs5 = str(models + 'general_spread_risk_state_rf_model_2025_NOENCODE.sav')
pickle.dump(general_state_2025_model, open('C:/Users/EK111/Documents/NJIT/Spotted Lanternfly Project URI Summer 2025/Deliverables/During Research/Models/state_updated_model_2025.sav', 'wb'))

In [ ]:
general_county_2026_model = DecisionTreeClassifier()

general_county_2026_model.fit(c6_split['X_train'], c6_split['Y_train'])
    
cv_results = cross_val_score(RandomForestClassifier(n_estimators = 100), c6_split['X_train'], c6_split['Y_train'], cv = 5, scoring = 'accuracy')
print('Average training-validation accuracy: ',cv_results.mean(),' Standard deviation of training-validation accuracy: ',cv_results.std())
    
print(classification_report(c6_split['Y_validation'], general_county_2026_model.predict(c6_split['X_validation'])))
print(confusion_matrix(c6_split['Y_validation'], general_county_2026_model.predict(c6_split['X_validation'])))
    
tt_results = cross_val_score(RandomForestClassifier(n_estimators = 100), c6_split['X_test'], c6_split['Y_test'], cv = 5, scoring = 'accuracy')
print('Average training-test accuracy: ',tt_results.mean(),' Standard deviation of training-test accuracy: ',tt_results.std())

print(classification_report(c6_split['Y_test'], general_county_2026_model.predict(c6_split['X_test'])))
print(confusion_matrix(c6_split['Y_test'], general_county_2026_model.predict(c6_split['X_test'])))

print('Predictions for counties in 2025:')

general_county_2026_predictions = general_county_2026_model.predict(c6_split['X_predict'])

general_county_2026_predictions_df = c6_split['X_predict']
general_county_2026_predictions_df['target'] = general_county_2026_predictions

general_county_2026_predictions_df.head()

In [ ]:
# path_grfc6 = str(models + 'general_spread_risk_county_rf_model_2026_NOENCODE.sav')
pickle.dump(general_county_2026_model, open('C:/Users/EK111/Documents/NJIT/Spotted Lanternfly Project URI Summer 2025/Deliverables/During Research/Models/county_updated_model_2026.sav', 'wb'))

In [ ]:
general_state_2026_model = DecisionTreeClassifier()

general_state_2026_model.fit(s5_split['X_train'], s5_split['Y_train'])
    
cv_results = cross_val_score(RandomForestClassifier(n_estimators = 100), s5_split['X_train'], s5_split['Y_train'], cv = 5, scoring = 'accuracy')
print('Average training-validation accuracy: ',cv_results.mean(),' Standard deviation of training-validation accuracy: ',cv_results.std())
    
print(classification_report(s5_split['Y_validation'], general_state_2026_model.predict(s5_split['X_validation'])))
print(confusion_matrix(s5_split['Y_validation'], general_state_2026_model.predict(s5_split['X_validation'])))
    
tt_results = cross_val_score(RandomForestClassifier(n_estimators = 100), s5_split['X_test'], s5_split['Y_test'], cv = 5, scoring = 'accuracy')
print('Average training-test accuracy: ',tt_results.mean(),' Standard deviation of training-test accuracy: ',tt_results.std())

print(classification_report(s5_split['Y_test'], general_state_2026_model.predict(s5_split['X_test'])))
print(confusion_matrix(s5_split['Y_test'], general_state_2026_model.predict(s5_split['X_test'])))

print('Predictions for states in 2025:')

general_state_2026_predictions = general_state_2026_model.predict(s5_split['X_predict'])

general_state_2026_predictions_df = s5_split['X_predict']
general_state_2026_mpredictions_df['target'] = general_state_2026_predictions

general_state_2026_predictions_df.head()

In [ ]:
# path_grfs6 = str(models + 'general_spread_risk_state_rf_model_2026_NOENCODE.sav')
pickle.dump(general_county_2026_model, open('C:/Users/EK111/Documents/NJIT/Spotted Lanternfly Project URI Summer 2025/Deliverables/During Research/Models/state_updated_model_2026.sav', 'wb'))

In [5]:
# Test splitting that guarantees at least some of every year is included in training, validation, and test sets
# Have to make a separate function from pd.sample because it has the chance to underselect for certain years

def split_50_30_20(dataframe: pd.DataFrame, index_cutoff: int = 3108, seed = 1) -> list:
    try: # only works if you have a 'year' column in the original and the clone
        by_year = {} # to assign the subsets of the dataframes per year
        # yearlength = {} # to assign the number of rows that exist per year
        
        for i in range(0, 11): # act for every year in the dataset; if ambiguous the end of the range would be len(pd.unique(basic_set['year']).tolist())
            # The below data adds rows of the year to the value of a dictionary where the key is the integer of the year
            by_year[i + 2014] = dataframe.loc[dataframe['year'] == i + 2014].reset_index() # makes the subset the same as the year, from 2014 to 2024
            # yearsize = 0
            
            # for j in range(0, len(basic_set)): # act for every row in the dataset; this is only necessary if I can't take the size of the year subset
            #     if basic_set.at[j, 'year'] == i + 2014: # act only if the year in the row is the same as the year in the outer loop iteration
            #         yearsize += 1 # tallies up the number of rows in the year
            # yearlength[pd.unique(basic_set['year'])[i]] = yearsize # adds data to the yearsize dictionary
            
        # Now split each year into small DataFrames and concatenate them into the three subsets for training, validation, and testing

        train_set = pd.DataFrame({'state': [], 'year': [], 'slf_pop': [], 'slfdensity': [], 'geometry': [], 'target': []})
        validation_set = pd.DataFrame({'state': [], 'year': [], 'slf_pop': [], 'slfdensity': [], 'geometry': [], 'target': []})
        test_set = pd.DataFrame({'state': [], 'year': [], 'slf_pop': [], 'slfdensity': [], 'geometry': [], 'target': []})
        # year_split_sizes = {}

        for i in range(0, 11): # act for every year in the dataset, if ambiguous the end of the range would be len(pd.unique(basic_set['year']).tolist())
            current_year_size = index_cutoff
            # current_year_size = yearlength[pd.unique(basic_set['year'])[i]]
            # year_size_train = m.ceil(0.5 * current_year_size) # round up to higher training set size in case split is uneven
            # year_size_validation = m.ceil(0.3 * current_year_size) # same for validation
            # year_size_test = current_year_size - year_size_train - year_size_validation
            # year_split_sizes[pd.unique(basic_set['year'])[i]] = [year_size_train, year_size_validation,
            #                                                      current_year_size - year_size_train - year_size_validation
            #                                                     ]
            try:
                new_seed = abs(int(m.floor(seed))) # converts any numerical seed to the floor value, then an integer if it's a float, and take the A.V.
                print('Seeded')
                
            except TypeError:
                print('The seed value passed to this function is not numerical.')

            else:
                new_seed = 1
                
            finally:
                if new_seed > 1:
                    new_seed = 1 / new_seed # returns the inverse of the seed if it is greater than 1
                # combinations_train = m.factorial(current_year_size) / (m.factorial(year_size_train) * m.factorial(current_year_size - year_size_train))
                # combinations_validation = m.factorial(current_year_size) / (m.factorial(year_size_validation) * 
                #                                                             m.factorial(current_year_size - year_size_validation))
                # combinations_test = m.factorial(current_year_size) / (m.factorial(year_size_test) * m.factorial(current_year_size - year_size_test))
                # combinations_total = m.factorial(current_year_size) / ()
                all_indices = by_year[i + 2014].index.tolist()
                index_list = list(range(0, index_cutoff))
                rand.shuffle(index_list) # add new_seed?
                print('Shuffled')
                # shuffled_indices = rand.shuffle(all_indices, new_seed)
                
                if current_year_size != len(index_list):
                    print('Some rows were lost in shuffling.')

                train_indices = index_list[0 : m.ceil(0.5 * len(index_list))]
                validation_indices = index_list[m.ceil(0.5 * len(index_list)) : m.ceil(0.8 * len(index_list))]
                test_indices = index_list[m.ceil(0.8 * len(index_list)) : len(index_list)]

                # Make this year's subsets use the indices generated for their purpose
                    
                train_this_year = by_year[i + 2014].iloc[train_indices]
                print('Training for year',i + 2014,'complete')
                validation_this_year = by_year[i + 2014].iloc[validation_indices]
                print('Validation for year',i + 2014,'complete')
                test_this_year = by_year[i + 2014].iloc[test_indices]
                print('Testing for year',i + 2014,'complete')

                # Add this year's splits to the main three split datasets
                
                train_set = pd.concat([train_set, train_this_year]) # train_set.merge(train_this_year, how = 'outer')
                print('Merging train set complete')
                validation_set = pd.concat([validation_set, validation_this_year]) # validation_set.merge(validation_this_year, how = 'outer')
                print('Merging validation set complete')
                test_this_year = pd.concat([test_set, test_this_year]) # test_set.merge(test_this_year, how = 'outer')
                print('Merging test set complete')
        
    except KeyError:
        print('Could not find the year column in the dataset.')
    else: # resorts to splitting the dataset without dealing with the years; if the dataframe was passed sorting earliest to latest years, this will
          # likely ruin a the capacity for a model to train successfully when involving the time factor
        train_set = dataframe.iloc[0 : m.ceil(len(dataframe) * 0.5)]
        validation_set = dataframe.iloc[m.ceil(len(dataframe) * 0.5) : m.ceil(len(dataframe) * 0.8)]
        test_set = dataframe.iloc[m.ceil(len(dataframe) * 0.8): len(dataframe)]
    finally:
        pass
        
    return [train_set, validation_set, test_set] # might have to convert each set into a dictionary and then reconvert into DataFrames if they can't be 
                                                 # called from a list


In [7]:
def choose(paramname: 'ponies', choices: list):
    print('Now you need to choose one from the following list of ',paramname,':\n')
    not_answered = True
    while not_answered:
        for i in range(0, len(choices)):
            print('- ',i,'\n')
        choice = input('Enter your choice of ',paramname,'on this line: ')
        try:
            if choice not in [str(i) for i in choices]:
                print('Your choice was not one of those available. Please choose from:\n')
            else:
                not_answered = False
        except Exception:
            pass
        finally:
            stringified = [''.join(str(i) for i in choices[j]) for j in choices] # might not work since the arrays are nested into the lists
            if choice not in stringified:
                print('Your choice was not one of those available. Please choose from:\n')
            else:
                not_answered = False
    return choice

def mlp_hyperparams(solver = 'lbfgs') -> dict:
    if solver != 'lbfgs' and solver != 'sgd' and solver != 'adam':
        print('Could not understand the Multi-Layer Perceptron solver type passed to this function. We will be using the \'lbfgs\' solver.')
        solver = 'lbfgs'

    print('Would you like to choose the basic recommended hyperparameters for low-data Multi-Layer Perceptrons? These are:\n',
          'Hidden layers: one hidden layer with five nodes\n',
          'Activation method: rectified linear (\'relu\'), meaning it takes the greater of 0 and the input to the layer\n',
          'Alpha value (regularizaiton value): 0.0001\n',
          'Maximum number of iterations: 200\n',
          'Random state (an integer seed that determines the outcome of the model fitting): 1\n',
          'Warm start (allows an iteration to use a previous solution to influence the solution): True'
         )
    menu = input('Enter your response (Y for yes or N for no) here: ')
    if str(menu).lower() == 'y' or str(menu).lower() == 'yes':
        choice_hl = [5]
        choice_ac = 'relu'
        choice_al = 0.0001
        choice_mi = 200
        choice_rs = 1
        choice_ws = True
        choice_bs, choice_lr, choice_li, choice_s, choice_p, choice_mo, choice_n, choice_b1, choice_b2, choice_e = -1

    else:
        # Add universal featuresets for all solvers
    
        hidden_layer_sizes = [[5], [5,1], [5,2], [5,3], [5,4], [5,5], [5,1,1], [5,1,2], [5,1,3], [5,1,4], [5,1,5], [5,2,1], [5,2,2], [5,2,3], [5,2,4], [5,2,5],
                     [5,3,1], [5,3,2], [5,3,3], [5,3,4], [5,3,5], [5,4,1], [5,4,2], [5,4,3], [5,4,4], [5,4,5], [5,5,1], [5,5,2], [5,5,3], [5,5,4], [5,5,5]
                    ]
        choice_hl = choice(paramname = 'hidden layer sizes', choices = hidden_layer_sizes)
        activation = ['identity', 'logistic', 'tanh', 'relu']
        choice_ac = choice(paramname = 'activation methods', choices = activation)
        alpha = [1e-6, 1e-5, 0.0001, 0.001, 0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1, 0.2, 0.3]
        choice_al = choice(paramname = 'alpha values', choices = alpha)
        max_iter = [50, 75, 100, 125, 150, 175, 200, 225, 250, 275, 300, 350]
        choice_mi = choice(paramname = 'maximum number of iterations', choices = max_iter)
        random_state = list(range(0, 1000))
        choice_rs = choice(paramname = 'seeds', choices = random_state)
        warm_start = [True, False]
        choice_ws = choice(paramname = 'allowance for the model to use some data from the previous iteration to save time', choices = warm_start)
    
        # Add extra featuresets for sgd and adam solvers
        
        if solver == 'sgd' or solver == 'adam':
            batch_size = [50, 100, 200, 300, 400, 500, 600, 700, 800, 900, 1000, 1100, 1200, 1300, 1400, 1500, 1600, 1700, 1800, 1900, 2000]
            choice_bs = choice(paramname = 'number of estimators per batch', choices = batch_size)
            learning_rate = ['constant', 'invscaling', 'adaptive']
            choice_lr = choice(paramname = 'model learning rates', choices = learning_rate)
            learning_rate_init = [1e-5, 0.0001, 0.001, 0.01, 0.2, 0.5]
            choice_li = choice(paramname = 'initial model learning rates', choices = learning_rate_init)
            shuffle = [True, False]
            choice_s = choice(paramname = 'shuffling of data', choices = shuffle)
        else:
            choice_bs, choice_lr, choice_li, choice_s = -1
    
        # Add extra featuresets for sgd solver
    
        if solver == 'sgd':
            power_t = [0.05, 0.1, 0.2, 0.5, 1, 2, 5]
            choice_p = choice(paramname = 'SGD solver inverse scaling values', choices = power_t)
            momentum = [i / 1000 for i in list(float(range(0, 1000)))]
            choice_mo = choice(paramname = 'SGD solver momentum', choices = momentum)
            nesterovs_momentum = [True, False]
            choice_n = choice(paramname = 'SGD solver Nesterov\'s momentum feature', choices = nesterovs_momentum)
        else:
            choice_p, choice_mo, choice_n = -1
    
        # Add extra featuresets for adam solver
    
        if solver == 'adam':
            beta_1 = [i / 1000 for i in list(float(range(0, 1000)))]
            choice_b1 = choice(paramname = 'first decay rate for the Adam solver', choices = beta_1)
            beta_2 = [i / 1000 for i in list(float(range(0, 1000)))]
            choice_b2 = choice(paramname = 'second decay rate for the Adam solver', choices = beta_2) 
            epsilon = [1e-6, 1e-7, 1e-8, 1e-9, 1e-10, 0]
            choice_e = choice(paramname = 'additional Epsilon value for the Adam solver', choices = epsilon) 
        else:
            choice_b1, choice_b2, choice_e = -1
        
    hyperparameters = {'hidden_layer_sizes': array(choice_hl), 'activation': str(choice_ac), 'solver': str(solver), 'alpha': float(choice_al),
                        'max_iter': int(choice_mi), 'random_state': int(choice_rs), 'warm_start': bool(choice_ws), 'batch_size': int(choice_bs),
                        'learning_rate': str(choice_lr), 'learning_rate_init': float(choice_li), 'shuffle': bool(choice_s), 'power_t': float(choice_p),
                        'momentum': float(choice_mo), 'nesterovs_momentum': bool(choice_n), 'beta_1': float(choice_b1), 'beta_2': float(choice_b2),
                        'epsilon': float(choice_e)
                       }
    return hyperparameters

def train_mlp(dataframe: pd.DataFrame, params: dict = {'solver': 'lbfgs', }, index_cutoff = 3108) -> list:
    # train Multi-Layer Perceptron; used for the artificial neural networks
    # hls = (5,), solver = 'lbfgs', significance_threshold = 1e-5, seed = 1      ...and activation?
    if params['solver'] == 'lbfgs':
        mlp_ann_classifier = MLPClassifier(solver = 'lbfgs', hidden_layer_sizes = params['hidden_layer_sizes'], activation = params['activation'],
                                           alpha = params['alpha'], max_iter = params['max_iter'], random_state = params['random_state'], warm_start = 
                                           params['warm_start']
                                          )

    elif params['solver'] == 'sgd': # stochastic gradient descent
        mlp_ann_classifier = MLPClassifier(solver = 'sgd', hidden_layer_sizes = params['hidden_layer_sizes'], activation = params['activation'],
                                           alpha = params['alpha'], max_iter = params['max_iter'], random_state = params['random_state'], warm_start = 
                                           params['warm_start'], batch_size = params['batch_size'], learning_rate = params['learning_rate'],
                                           learning_rate_init = params['learning_rate_init'], shuffle = params['shuffle'], power_t = params['power_t'],
                                           momentum = params['momentum'], nesterovs_momentum = ['nesterovs_momentum']
                                          )

    elif params['solver'] == 'adam':
        mlp_ann_classifier = MLPClassifier(solver = 'adam', hidden_layer_sizes = params['hidden_layer_sizes'], activation = params['activation'],
                                           alpha = params['alpha'], max_iter = params['max_iter'], random_state = params['random_state'], warm_start = 
                                           params['warm_start'], batch_size = params['batch_size'], learning_rate = params['learning_rate'],
                                           learning_rate_init = params['learning_rate_init'], shuffle = params['shuffle'], beta_1 = params['beta_1'],
                                           beta_2 = params['beta_2'], epsilon = params['epsilon']
                                          )

    else:
        print('The solver you want to use for this model could not be found. We will be using the lbfgs solver instead.')
        mlp_ann_classifier = MLPClassifier(solver = 'lbfgs', hidden_layer_sizes = params['hidden_layer_sizes'], activation = params['activation'],
                                           alpha = params['alpha'], max_iter = params['max_iter'], random_state = params['random_state'], warm_start = 
                                           params['warm_start']
                                          )
    splits = split_50_30_20(dataframe = pd.DataFrame, index_cutoff = 3108, seed = 1)
    X_train = splits[0].drop(columns = ['target', 'geometry'], axis = 1)
    Y_train = splits[0]['target']
    X_validation = splits[1].drop(columns = ['target', 'geometry'], axis = 1)
    Y_validation = splits[1]['target']
    X_test = splits[2].drop(columns = ['target', 'geometry'], axis = 1)
    Y_test = splits[2]['target']
        
    mlp_ann_classifier.fit(dataframe.drop(['geometry', 'target'], axis = 1), dataframe['target'])

    cv_results = cross_val_score(mlp_ann_classifier, X_train, Y_train, cv = 1, scoring = 'accuracy')
    print('Average training-validation accuracy: ',cv_results.mean(),' Standard deviation of training-validation accuracy: ',cv_results.std())

    print(classification_report(Y_validation, dtree.predict(X_validation)))
    print(confusion_matrix(Y_validation, dtree.predict(X_validation)))
    
    tt_results = cross_val_score(mlp_ann_classifier, X_test, Y_test, cv = 1, scoring = 'accuracy')
    print('Average training-test accuracy: ',tt_results.mean(),' Standard deviation of training-test accuracy: ',tt_results.std())

    print(classification_report(Y_test, dtree.predict(X_test)))
    print(confusion_matrix(Y_test, dtree.predict(X_test)))
    
    return {'training_features': X_train, 'training_target': Y_train, 'validation_features': X_validation, 'validation_target': Y_validation,
            'test_features': X_test, 'test_target': Y_test, 'validation_results_mean': cv_results.mean(), 'validation_results_std': cv_results.std(),
            'test_results_mean': tt_results.mean(), 'test_results_std': cv_results.std(), 'model': mlp_ann_classifier}

In [9]:
def preprocess_rf(dataframe: pd.DataFrame, index_cutoff: int = 3108, seed_split: int = 1) -> list:
        # X_train, X_validation, Y_train, Y_validation = train_test_split(dataframe.drop(['target','geometry'], axis = 1), dataframe['target'], test_size = 0.20, random_state = 1, shuffle = True)
        encoder = OneHotEncoder(handle_unknown = "ignore") # doing this because training set, validation set, and test set have different number of features
        defragmented_df = dataframe.copy()
        base = encoder.fit_transform(defragmented_df.drop(columns = ['year', 'target', 'geometry'], axis = 1))
        base_df = pd.DataFrame(base.toarray())
        base_df['year'] = dataframe['year'] # add year back in because it is important to the splitting process
        base_df['target'] = dataframe['target']
        print('Preprocessing complete. Splitting...')
        return split_50_30_20(base_df, index_cutoff, seed_split)

In [129]:
def my_encoder_v2(df: pd.DataFrame) -> pd.DataFrame:
    encoded = pd.DataFrame({})
    columns = df.columns.tolist()
    for i in range(0, len(columns)):
        this_column = columns[i]
        this_column_as_list = df[this_column].tolist()
        print('Processing the column',this_column)
        encoded_items = []
        columntype = type(df.at[0, this_column])
        
        if columntype == str and this_column != 'target':
            encoded_items = OrdinalEncoder().fit_transform(this_column_as_list)

        elif this_column == 'target':
            encoded_items = LabelEncoder().fit_transform(this_column_as_list)
                
        elif columntype == int or columntype == np.int64:
            encoded_items = OneHotEncoder().fit_transform(this_column_as_list)
            
        else:
            print('Column \'',this_column,'\' cannot be encoded')
            encoded_items = this_column_as_list
            
        encoded[str(i)] = encoded_items # might have to convert the string column labels into integer column labels
    return encoded

In [29]:
splits_county_no_encoding = split_50_30_20(dataframe = county_gdf, index_cutoff = 3108, seed = 1)

Seeded
Shuffled
Training for year 2014 complete
Validation for year 2014 complete
Testing for year 2014 complete
Merging train set complete
Merging validation set complete
Merging test set complete
Seeded
Shuffled
Training for year 2015 complete
Validation for year 2015 complete
Testing for year 2015 complete
Merging train set complete
Merging validation set complete
Merging test set complete
Seeded
Shuffled
Training for year 2016 complete
Validation for year 2016 complete
Testing for year 2016 complete
Merging train set complete
Merging validation set complete
Merging test set complete
Seeded
Shuffled
Training for year 2017 complete
Validation for year 2017 complete
Testing for year 2017 complete
Merging train set complete
Merging validation set complete
Merging test set complete
Seeded
Shuffled
Training for year 2018 complete
Validation for year 2018 complete
Testing for year 2018 complete
Merging train set complete
Merging validation set complete
Merging test set complete
Seeded
Shu

In [31]:
splits_state_no_encoding = split_50_30_20(dataframe = state_gdf, index_cutoff = 48, seed = 1)

Seeded
Shuffled
Training for year 2014 complete
Validation for year 2014 complete
Testing for year 2014 complete
Merging train set complete
Merging validation set complete
Merging test set complete
Seeded
Shuffled
Training for year 2015 complete
Validation for year 2015 complete
Testing for year 2015 complete
Merging train set complete
Merging validation set complete
Merging test set complete
Seeded
Shuffled
Training for year 2016 complete
Validation for year 2016 complete
Testing for year 2016 complete
Merging train set complete
Merging validation set complete
Merging test set complete
Seeded
Shuffled
Training for year 2017 complete
Validation for year 2017 complete
Testing for year 2017 complete
Merging train set complete
Merging validation set complete
Merging test set complete
Seeded
Shuffled
Training for year 2018 complete
Validation for year 2018 complete
Testing for year 2018 complete
Merging train set complete
Merging validation set complete
Merging test set complete
Seeded
Shu

In [11]:
splits_county = preprocess_rf(dataframe = county_gdf, index_cutoff = 3108, seed_split = 1)
splits_county[0].head()

Preprocessing complete. Splitting...
Seeded
Shuffled
Training for year 2014 complete
Validation for year 2014 complete
Testing for year 2014 complete
Merging train set complete
Merging validation set complete
Merging test set complete
Seeded
Shuffled
Training for year 2015 complete
Validation for year 2015 complete
Testing for year 2015 complete
Merging train set complete
Merging validation set complete
Merging test set complete
Seeded
Shuffled
Training for year 2016 complete
Validation for year 2016 complete
Testing for year 2016 complete
Merging train set complete
Merging validation set complete
Merging test set complete
Seeded
Shuffled
Training for year 2017 complete
Validation for year 2017 complete
Testing for year 2017 complete
Merging train set complete
Merging validation set complete
Merging test set complete
Seeded
Shuffled
Training for year 2018 complete
Validation for year 2018 complete
Testing for year 2018 complete
Merging train set complete
Merging validation set complete

,0,1,2,3,4,5,6,7,8,9,...,10155,10156,10157,10158,10159,10160,10161,10162,year,target
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,2014,0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,2014,0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,2014,0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,2014,0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,2014,0


In [15]:
splits_state = preprocess_rf(dataframe = state_gdf, index_cutoff = 48, seed_split = 1)
splits_state[0].head()

Preprocessing complete. Splitting...
Seeded
Shuffled
Training for year 2014 complete
Validation for year 2014 complete
Testing for year 2014 complete
Merging train set complete
Merging validation set complete
Merging test set complete
Seeded
Shuffled
Training for year 2015 complete
Validation for year 2015 complete
Testing for year 2015 complete
Merging train set complete
Merging validation set complete
Merging test set complete
Seeded
Shuffled
Training for year 2016 complete
Validation for year 2016 complete
Testing for year 2016 complete
Merging train set complete
Merging validation set complete
Merging test set complete
Seeded
Shuffled
Training for year 2017 complete
Validation for year 2017 complete
Testing for year 2017 complete
Merging train set complete
Merging validation set complete
Merging test set complete
Seeded
Shuffled
Training for year 2018 complete
Validation for year 2018 complete
Testing for year 2018 complete
Merging train set complete
Merging validation set complete

,0,1,2,3,4,5,6,7,8,9,...,403,404,405,406,407,408,409,410,year,target
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2014,0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2014,0
2,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2014,0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2014,0
4,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2014,0


In [17]:
splits_county[0].drop(columns = ['target'], axis = 1).head()

,0,1,2,3,4,5,6,7,8,9,...,10154,10155,10156,10157,10158,10159,10160,10161,10162,year
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,2014
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,2014
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,2014
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,2014
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,2014


In [19]:
def train_rf(splits: list, seed_split: int = 1, estimators: int = 100) -> dict: # train a Random Forest model; used for the decision tree
                                                                                           # ensemble. parameter seed_fold = 1 may be needed if I need to
                                                                                           # reintroduce the kfold
    print('Splitting complete. Getting training set...')
    # X_train = splits[0].drop(columns = ['target', 'geometry'], axis = 1)
    # X_train_encoded = encoder.fit_transform(X_train)
    X_train = splits[0].drop(columns = ['target'], axis = 1)
    # X_train[9999] = X_train['year']                          I don't know why I decided to both add and drop the year column
    X_train = X_train.drop(columns = ['year'], axis = 1)
    Y_train = splits[0]['target']
    print('Train set complete')
    
    # X_validation = splits[1].drop(columns = ['target', 'geometry'], axis = 1)
    # X_validation_encoded = encoder.fit_transform(X_validation)
    X_validation = splits[1].drop(columns = ['target'], axis = 1)
    # X_validation[9999] = X_validation['year']
    X_validation = X_validation.drop(columns = ['year'], axis = 1)
    Y_validation = splits[1]['target']
    print('Validation set complete')
    
    # X_test = splits[2].drop(columns = ['target', 'geometry'], axis = 1)
    # X_test_encoded = encoder.fit_transform(X_test)
    X_test = splits[2].drop(columns = ['target'], axis = 1)
    # X_test[9999] = X_test['year']
    X_test = X_test.drop(columns = ['year'], axis = 1)
    Y_test = splits[2]['target']
    print('Test set complete')
    
    # kfold = StratifiedKFold(n_splits = 10, random_state = seed_fold, shuffle = True); could be passed to parameter cv in cross_val_score()
    # cv = 5 in cv_results and tt_results because there are five classes

    dtree = DecisionTreeClassifier()

    # for i in range(0, len())   was going to use a loop to encode but it would have been a bad time

    print('Fitting...')
    
    dtree.fit(X_train, Y_train)
    
    print('Fitting complete. Cross-validating...')
    
    cv_results = cross_val_score(RandomForestClassifier(n_estimators = estimators), X_train, Y_train, cv = 5, scoring = 'accuracy')
    print('Average training-validation accuracy: ',cv_results.mean(),' Standard deviation of training-validation accuracy: ',cv_results.std())
    
    print('Predicting based on validation set...')
    
    print(classification_report(Y_validation, dtree.predict(X_validation)))
    print(confusion_matrix(Y_validation, dtree.predict(X_validation)))
    
    print('Cross validation based on test set...')
    
    tt_results = cross_val_score(RandomForestClassifier(n_estimators = estimators), X_test, Y_test, cv = 5, scoring = 'accuracy')
    print('Average training-test accuracy: ',tt_results.mean(),' Standard deviation of training-test accuracy: ',tt_results.std())

    print('Predicting based on test set...')
    
    print(classification_report(Y_test, dtree.predict(X_test)))
    print(confusion_matrix(Y_test, dtree.predict(X_test)))

    return {'training_features': X_train, 'training_target': Y_train, 'validation_features': X_validation, 'validation_target': Y_validation,
            'test_features': X_test, 'test_target': Y_test, 'validation_results_mean': cv_results.mean(), 'validation_results_std': cv_results.std(),
            'test_results_mean': tt_results.mean(), 'test_results_std': cv_results.std(), 'tree': dtree
           }

In [21]:
def train_multiple_rf(dataframe: pd.DataFrame, split_seeds: int = 5, estimators_per_forest: int = 100) -> list:
# parameter fold_seeds = 5 may need to be added if kfold is needed
    if type(split_seeds) != int or split_seeds < 1:
        print('The number of seeds for dataset splitting was not passed as a parseable number. We will work with five seeds, or five attempts to train ',
              'the Random Forest Classifier.')
        split_seeds = list(range(0, 5))
    
    results = []
    
    for i in range(0, split_seeds):
        results = results + [train_rf(dataframe, seed_split = i, estimators = estimators_per_forest)]

    # compare results

    print('RESULTS:\nSeeds used: ',range(0, split_seeds),'\n')

    accuracy_cv_mean = [results[i]['validation_results_mean'] for i in results]
    accuracy_cv_std = [results[i]['validation_results_std'] for i in results]
    accuracy_tt_mean = [results[i]['test_results_mean'] for i in results]
    accuracy_tt_std = [results[i]['test_results_std'] for i in results]

    # evaluate most accurate and most precise results
    
    greatest_cv_mean = [0] # evaluates accuracy in cross-validation
    least_cv_std = [9999] # evaluates precision in cross-validation
    greatest_tt_mean = [0] # evaluates accuracy in train-test
    least_tt_std = [999] # evaluates precision in train-test

    for i in range(0, split_seeds):
        if greatest_cv_mean[0] < accuracy_cv_mean[i]:
            greatest_cv_mean[0] = accuracy_cv_mean[i] # replaces the score
            greatest_cv_mean = greatest_cv_mean + [i] # adds the seed to the right side of the list
            greatest_cv_mean = greatest_cv_mean[0:2] # removes any tied seeds
        elif greatest_cv_mean[0] == accuracy_cv_mean[i]:
            greatest_cv_mean = greatest_cv_mean.append([str('and ',i)])
                                                           
        if least_cv_std[0] > accuracy_cv_std[i]:
            least_cv_std[0] = accuracy_cv_mean[i]
            least_cv_std = least_cv_std + [i]
        elif least_cv_std[0] == accuracy_cv_std[i]:
            least_cv_std = least_cv_std.append([str('and ',i)])
            
        if greatest_tt_mean[0] < accuracy_tt_mean[i]:
            greatest_tt_mean[0] = accuracy_cv_mean[i]
            greatest_tt_mean = greatest_tt_mean + [i]
        elif greatest_tt_mean[0] == accuracy_tt_mean[i]:
            greatest_tt_mean = greatest_tt_mean.append([str('and ',i)])
            
        if least_tt_std[0] > accuracy_tt_std[i]:
            least_tt_std[0] = accuracy_cv_mean[i]
            least_tt_std = least_tt_std + [i]
        elif least_tt_std[0] == accuracy_tt_std[i]:
            least_tt_std = least_tt_std.append([str('and ',i)])
        
    print('Greatest average cross-validation score: ',greatest_cv_mean[0],'\nSeed(s) with greatest cross-validation score on average: ',
          greatest_cv_mean[1:],'\nMost precise cross-validation standard deviation (lowest standard deviation): ',least_cv_std[0],'\nSeed(s) with the ',
          'most precise cross-validation standard deviation: ',least_cv_std[1:],'\nGreatest average train-test score: ',greatest_tt_mean[0],
          '\nSeed(s) with the greatest train-test score on average: ',greatest_tt_mean[1:],'\nMost precise train-test standard deviation: ',
          least_tt_std[0],'\nSeed(s) with the most precise train-test standard deviation: ',least_tt_std[1:]
         )
    
    
    return results

In [61]:
splits_county_no_encoding[0].columns

Index([    'county',      'state',  'land_area', 'water_area',       'year',
          'slf_pop', 'slf_pop_ye', 'slfdensity', 'y_slfdnsty',     'target',
       ...
         1449563753,   1201239500,   1749093456,   1966281375,   1512221259,
         1057037102,   2409688595,   1592697307,   1151832460,   1031795759],
      dtype='object', length=3627)

In [59]:
this_split_county_pre_encoding = splits_county_no_encoding[0].copy()
train_frame = this_split_county_pre_encoding # dataframe
train_columns = train_frame.columns.tolist() # list of strings
splits_county_soft_encoding = [pd.DataFrame({}), pd.DataFrame({}), pd.DataFrame({})] # list of dataframes
train_soft_encoding = splits_county_soft_encoding[0]
for j in range(0, len(train_columns)): # iterates through a list of strings
    current_column = train_frame[train_columns[j]] # column of dataframe, given a string label
    train_soft_encoding[current_column] = pd.to_numeric(current_column, errors = 'coerce') # column of dataframe, given an altered column

C:\Users\EK111\AppData\Local\Temp\ipykernel_45404\3047034362.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_soft_encoding[current_column] = pd.to_numeric(current_column, errors = 'coerce') # column of dataframe, given an altered column
C:\Users\EK111\AppData\Local\Temp\ipykernel_45404\3047034362.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_soft_encoding[current_column] = pd.to_numeric(current_column, errors = 'coerce') # column of dataframe, given an altered column
C:\Users\EK111\AppData\Local\Temp\ipyker

KeyboardInterrupt: 

In [49]:
for i in range(0, 3):
    this_split_county_pre_encoding = splits_county_no_encoding[i].copy()
    this_frame = splits_county_pre_encoding # dataframe
    these_columns = this_frame.columns.tolist() # list of strings
    splits_county_soft_encoding = [pd.DataFrame({}), pd.DataFrame({}), pd.DataFrame({})] # list of dataframes
    this_frame_soft_encoding = splits_county_soft_encoding[i]
    for j in range(0, len(these_columns)): # iterates through a list of strings
            current_column = this_frame[these_columns[j]] # column of dataframe, given a string label
            this_frame_soft_encoding[current_column] = pd.to_numeric(current_column, errors = 'coerce') # column of dataframe, given an altered column
# for i in range(0, len(splits_state_no_encoding.columns.tolist())):
#     splits_state_soft_encoding = pd.DataFrame({})
#     current_column = splits_state_no_encoding[splits_state_no_encoding.columns.tolist()[i]]
#     splits_state_soft_encoding[current_column] = to_numeric(splits_state_no_encoding[current_column], errors = 'coerce')

C:\Users\EK111\anaconda3\Lib\site-packages\geopandas\geodataframe.py:1819: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)
C:\Users\EK111\anaconda3\Lib\site-packages\geopandas\geodataframe.py:1819: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)
C:\Users\EK111\anaconda3\Lib\site-packages\geopandas\geodataframe.py:1819: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer]

KeyboardInterrupt: 

In [ ]:
for i in range(0, 3):
    this_frame = splits_state_no_encoding[i]
    these_columns = this_frame.columns.tolist()
    splits_state_soft_encoding = [pd.DataFrame({}), pd.DataFrame({}), pd.DataFrame({})]
    for j in range(0, len(these_columns)):
            current_column = splits_state_no_encoding[i][these_columns[j]]
            this_frame[current_column] = pd.to_numeric(splits_state_no_encoding[current_column], errors = 'coerce')

In [37]:
# splits_county_soft_encoding = splits_county_no_encoding.apply(pd.to_numeric, errors='coerce')
# splits_state_soft_encoding = splits_state_no_encoding.apply(pd.to_numeric, errors='coerce')

In [33]:
general_rf_c_no_encoding = train_rf(splits_county_soft_encoding, seed_split = 1, estimators = 100)

Splitting complete. Getting training set...
Train set complete
Validation set complete
Test set complete
Fitting...


ValueError: could not convert string to float: 'Houston'

In [23]:
# Run Random Forest on counties

grfc = train_rf(splits_county, seed_split = 1, estimators = 100)

Splitting complete. Getting training set...
Train set complete
Validation set complete
Test set complete
Fitting...
Fitting complete. Cross-validating...
Average training-validation accuracy:  1.0  Standard deviation of training-validation accuracy:  0.0
Predicting based on validation set...


C:\Users\EK111\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\EK111\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\EK111\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


              precision    recall  f1-score   support

           0       1.00      1.00      1.00      9956
           1       1.00      0.98      0.99       184
           3       0.00      0.00      0.00         1
           4       0.99      1.00      1.00       116

    accuracy                           1.00     10257
   macro avg       0.75      0.74      0.75     10257
weighted avg       1.00      1.00      1.00     10257

[[9956    0    0    0]
 [   4  180    0    0]
 [   0    0    0    1]
 [   0    0    0  116]]
Cross validation based on test set...


C:\Users\EK111\anaconda3\Lib\site-packages\sklearn\model_selection\_split.py:725: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


Average training-test accuracy:  0.9997076023391813  Standard deviation of training-test accuracy:  0.0005847953216374435
Predicting based on test set...


C:\Users\EK111\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\EK111\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\EK111\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


              precision    recall  f1-score   support

           0       1.00      1.00      1.00      6373
           1       1.00      0.94      0.97       340
           3       0.00      0.00      0.00         1
           4       0.99      1.00      1.00       123

    accuracy                           1.00      6837
   macro avg       0.75      0.73      0.74      6837
weighted avg       1.00      1.00      1.00      6837

[[6373    0    0    0]
 [  22  318    0    0]
 [   0    0    0    1]
 [   0    0    0  123]]


In [25]:
# Run Random Forest on states

grfs = train_rf(splits_state, seed_split = 1, estimators = 100)

Splitting complete. Getting training set...
Train set complete
Validation set complete
Test set complete
Fitting...
Fitting complete. Cross-validating...


C:\Users\EK111\anaconda3\Lib\site-packages\sklearn\model_selection\_split.py:725: UserWarning: The least populated class in y has only 4 members, which is less than n_splits=5.
  warnings.warn(


Average training-validation accuracy:  0.9924528301886791  Standard deviation of training-validation accuracy:  0.015094339622641506
Predicting based on validation set...
              precision    recall  f1-score   support

           0       0.99      1.00      1.00       117
           1       0.96      0.96      0.96        27
           3       0.00      0.00      0.00         1
           4       1.00      1.00      1.00        14

    accuracy                           0.99       159
   macro avg       0.74      0.74      0.74       159
weighted avg       0.98      0.99      0.98       159

[[117   0   0   0]
 [  1  26   0   0]
 [  0   1   0   0]
 [  0   0   0  14]]
Cross validation based on test set...


C:\Users\EK111\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\EK111\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\EK111\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\EK111\anaconda3\Lib\site-packages\sklearn\mode

Average training-test accuracy:  0.9523809523809523  Standard deviation of training-test accuracy:  0.05216405309573012
Predicting based on test set...
              precision    recall  f1-score   support

           0       0.90      1.00      0.95        54
           1       0.92      0.85      0.88        40
           2       0.00      0.00      0.00         1
           4       1.00      0.80      0.89        10

    accuracy                           0.91       105
   macro avg       0.70      0.66      0.68       105
weighted avg       0.91      0.91      0.91       105

[[54  0  0  0]
 [ 6 34  0  0]
 [ 0  1  0  0]
 [ 0  2  0  8]]


C:\Users\EK111\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\EK111\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\EK111\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [ ]:
# Run Multi-Layer Perceptron on counties

gmlpc = train_mlp()

In [ ]:
# Run Multi-Layer Perceptron on states

gmlps = train_mlp()

In [27]:
# Save these models using Pickle

models = 'C:/Users/EK111/Documents/NJIT/Spotted Lanternfly Project URI Summer 2025/Deliverables/During Research/Models/'

path_grfc = str(models + 'general_spread_risk_county_rf_model.sav')
path_grfs = str(models + 'general_spread_risk_state_rf_model.sav')
# path_gmlpc = str(models + 'general_spread_risk_county_mlp_model.sav')
# path_gmlps = str(models + 'general_spread_risk_state_mlp_model.sav')

pickle.dump(grfc['tree'], open('C:/Users/EK111/Documents/NJIT/Spotted Lanternfly Project URI Summer 2025/Deliverables/During Research/Models/original_county.sav', 'wb'))
pickle.dump(grfc, open(path_grfc, 'wb')) # Pickle uses the 'wb' or Write Binary mode to write files to the device
pickle.dump(grfs['tree'], open('C:/Users/EK111/Documents/NJIT/Spotted Lanternfly Project URI Summer 2025/Deliverables/During Research/Models/original_state.sav', 'wb'))
pickle.dump(grfs, open(path_grfs, 'wb'))
# pickle.dump(gmlpc, open(path_gmlpc, 'wb'))
# pickle.dump(gmlps, open(path_gmlps, 'wb'))

In [ ]:
# Modify data for food models

county_food_specific = gpd.read_file('C:/Users/EK111/Documents/NJIT/Spotted Lanternfly Project URI Summer 2025/Modified Data Backups/food_county.csv')

county_food_gdf = county_gdf
county_food_gdf['est_food_abundance'] = county_food_specific['est_food_abundance']
county_food_gdf['food_host_species_diversity'] = county_food_specific['food_host_species_diversity']
county_food_gdf['non_food_host_species_diversity'] = county_food_specific['non_food_host_species_diversity']

state_food_specific = gpd.read_file('C:/Users/EK111/Documents/NJIT/Spotted Lanternfly Project URI Summer 2025/Modified Data Backups/food_state.csv')

state_food_gdf = state_gdf
state_food_gdf['est_food_abundance'] = state_food_specific['est_food_abundance']
state_food_gdf['food_host_species_diversity'] = state_food_specific['food_host_species_diversity']
state_food_gdf['non_food_host_species_diversity'] = state_food_specific['non_food_host_species_diversity']

In [ ]:
# Run food damage risk Random Forest for counties

foodrfc = train_rf(county_food_gdf, seed_split = 1, estimators = 100)

# Run food damage risk Random Forest for states

foodrfs = train_rf(state_food_gdf, seed_split = 1, estimators = 100)

# Run food damage risk Multi-Layer Perceptron for counties

foodmlpc = train_mlp()

# Run food damage risk Multi-Layer Perceptron for states

foodmlps = train_mlp()

In [ ]:
# Save these models using Pickle

path_foodrfc = str(models + 'food_host_damage_risk_county_rf_model.sav')
path_foodrfs = str(models + 'food_host_damage_risk_state_rf_model.sav')
path_foodmlpc = str(models + 'food_host_damage_risk_county_mlp_model.sav')
path_foodmlps = str(models + 'food_host_damage_risk_state_mlp_model.sav')

pickle.dump(foodrfc, open(path_foodrfc, 'wb')) # Pickle uses the 'wb' or Write Binary mode to write files to the device
pickle.dump(foodrfs, open(path_foodrfs, 'wb'))
pickle.dump(foodmlpc, open(path_foodmlpc, 'wb'))
pickle.dump(foodmlps, open(path_foodmlps, 'wb'))

In [ ]:
# Modify data for timber and paper models

county_fiber_specific = gpd.read_file('C:/Users/EK111/Documents/NJIT/Spotted Lanternfly Project URI Summer 2025/Modified Data Backups/timber_paper_county.csv')

county_fiber_gdf = county_gdf
county_fiber_gdf['est_fiber_abundance'] = county_fiber_specific['est_fiber_abundance']
county_fiber_gdf['fiber_host_species_diversity'] = county_fiber_specific['fiber_host_species_diversity']
county_fiber_gdf['non_fiber_host_species_diversity'] = county_fiber_specific['non_fiber_host_species_diversity']

state_fiber_specific = gpd.read_file('C:/Users/EK111/Documents/NJIT/Spotted Lanternfly Project URI Summer 2025/Modified Data Backups/timber_paper_state.csv')

state_fiber_gdf = state_gdf
state_fiber_gdf['est_fiber_abundance'] = state_fiber_specific['est_fiber_abundance']
state_fiber_gdf['fiber_host_species_diversity'] = state_fiber_specific['fiber_host_species_diversity']
state_fiber_gdf['non_fiber_host_species_diversity'] = state_fiber_specific['non_fiber_host_species_diversity']

In [ ]:
# Run food damage risk Random Forest for counties

fiberrfc = train_rf(county_fiber_gdf, seed_split = 1, estimators = 100)

# Run food damage risk Random Forest for states

fiberrfs = train_rf(state_fiber_gdf, seed_split = 1, estimators = 100)

# Run food damage risk Multi-Layer Perceptron for counties

fibermlpc = train_mlp()

# Run food damage risk Multi-Layer Perceptron for states

fibermlps = train_mlp()

In [ ]:
# Save these models using Pickle

path_fiberrfc = str(models + 'fiber_host_damage_risk_county_rf_model.sav')
path_fiberrfs = str(models + 'fiber_host_damage_risk_state_rf_model.sav')
path_fibermlpc = str(models + 'fiber_host_damage_risk_county_mlp_model.sav')
path_fibermlps = str(models + 'fiber_host_damage_risk_state_mlp_model.sav')

pickle.dump(fiberrfc, open(path_fiberrfc, 'wb')) # Pickle uses the 'wb' or Write Binary mode to write files to the device
pickle.dump(fiberrfs, open(path_fiberrfs, 'wb'))
pickle.dump(fibermlpc, open(path_fibermlpc, 'wb'))
pickle.dump(fibermlps, open(path_fibermlps, 'wb'))

In [ ]:
# Modify data for ornamentals' models

county_ornamental_specific = gpd.read_file('C:/Users/EK111/Documents/NJIT/Spotted Lanternfly Project URI Summer 2025/Modified Data Backups/ornamental_county.csv')

county_ornamental_gdf = county_gdf
county_ornamental_gdf['est_ornamental_abundance'] = county_ornamental_specific['est_ornamental_abundance']
county_ornamental_gdf['ornamental_host_species_diversity'] = county_ornamental_specific['ornamental_host_species_diversity']
county_ornamental_gdf['non_ornamental_host_species_diversity'] = county_ornamental_specific['non_ornamental_host_species_diversity']

state_ornamental_specific = gpd.read_file('C:/Users/EK111/Documents/NJIT/Spotted Lanternfly Project URI Summer 2025/Modified Data Backups/ornamental_state.csv')

state_ornamental_gdf = state_gdf
state_ornamental_gdf['est_ornamental_abundance'] = state_ornamental_specific['est_ornamental_abundance']
state_ornamental_gdf['ornamental_host_species_diversity'] = state_ornamental_specific['ornamental_host_species_diversity']
state_ornamental_gdf['non_ornamental_host_species_diversity'] = state_ornamental_specific['non_ornamental_host_species_diversity']

In [ ]:
# Run food damage risk Random Forest for counties

orfc = train_rf(county_ornamental_gdf, seed_split = 1, estimators = 100)

# Run food damage risk Random Forest for states

orfs = train_rf(state_ornamental_gdf, seed_split = 1, estimators = 100)

# Run food damage risk Multi-Layer Perceptron for counties

omlpc = train_mlp()

# Run food damage risk Multi-Layer Perceptron for states

omlps = train_mlp()

In [ ]:
# Save these models using Pickle

path_orfc = str(models + 'ornamental_host_damage_risk_county_rf_model.sav')
path_orfs = str(models + 'ornamental_host_damage_risk_state_rf_model.sav')
path_omlpc = str(models + 'ornamental_host_damage_risk_county_mlp_model.sav')
path_omlps = str(models + 'ornamental_host_damage_risk_state_mlp_model.sav')

pickle.dump(orfc, open(path_orfc, 'wb')) # Pickle uses the 'wb' or Write Binary mode to write files to the device
pickle.dump(orfs, open(path_orfs, 'wb'))
pickle.dump(omlpc, open(path_omlpc, 'wb'))
pickle.dump(omlps, open(path_omlps, 'wb'))